# AI大模型学习笔记

## 整体技术链路

![Overall Technical Workflow](Overall_Technical_Workflow.png)

基础概念：模型是什么  
这一节回答最根本的问题：大模型到底是个什么东西，它靠什么工作。  
  
LLM（大语言模型）  
LLM 全称 Large Language Model，是基于海量文本训练出来的深度学习模型。  
  
它的本质是一个「预测下一个词」的概率模型：给定前面的文字，预测接下来最可能出现的内容。  
  
ChatGPT、Claude、通义千问等产品，底层都是 LLM。  
  
Transformer 与注意力机制  
几乎所有现代 LLM 都基于 Transformer 架构。  

它的核心创新是注意力机制（Attention / Self-Attention）：模型在处理某个词时，能动态地「关注」输入序列中其他相关的词，从而更好地理解上下文关系。  

打个比方，模型读一句话时不是死板地逐字扫描，而是会自动判断「这个词和前面哪个词关系最大」。  
  
Token（词元）  
模型并不按「字」或「单词」处理文本，而是按 Token 处理。  
  
一个 Token 可能是一个汉字、一个英文单词，也可能是单词的一部分。  

In [1]:
# 安装：pip install tiktoken
import tiktoken

# 不同模型使用不同分词器，encoding_for_model 会自动选对应的
enc = tiktoken.encoding_for_model("gpt-4o-mini")

tokens = enc.encode("Hello world")  # 把文本切成 Token 列表
print("Token 数量：", len(tokens))
print("切分结果：", [enc.decode([t]) for t in tokens])

Token 数量： 2
切分结果： ['Hello', ' world']


上下文窗口（Context Window）  
指模型一次能「看到」和处理的最大 Token 数量。  
  
比如上下文窗口是 100K，意味着一次对话（历史消息、你的问题、模型回答加在一起）不能超过 10 万个 Token。  
  
窗口越大，模型能「记住」的内容越多，能一次性处理的文档也越长。  
  
参数量（Parameters）  
参数量指模型内部可学习的权重数量，通常以 B（十亿，Billion）为单位，比如 7B、70B。  
  
它在一定程度上反映模型的规模与能力上限，但不是唯一指标——训练数据质量、架构设计同样关键。    
  
![Parameters](Parameters_table.png)

## 模型是如何训练出来的
一个能自然对话的 LLM，通常要经过多个训练阶段，层层叠加能力。

![The LLM Training Process](The_LLM_Training_Process.png)

预训练（Pre-training）  
模型在海量、无标注的文本（网页、书籍、代码等）上学习语言的统计规律，比如语法、常识、逻辑关系。  
  
这个阶段耗费的算力最大，是模型能力的基础。  
  
但此时的模型还不擅长「按指令做事」，更像一个博览群书却不懂如何交流的人。  
  
SFT（监督微调）  
SFT 全称 Supervised Fine-Tuning。  
  
用人工标注好的「问题—答案」对数据，教模型学会按照人类指令来回答问题，而不是自顾自地续写文本。  
  
RLHF（人类反馈强化学习）  
RLHF 全称 Reinforcement Learning from Human Feedback。  
  
让人类对模型的多个候选回答打分或排序，再用这些反馈训练模型，使其输出更符合人类偏好（更有用、更诚实、更安全）。  

类似的技术还有 RLAIF：用 AI 代替人类打分，以降低标注成本。

两者目的一致，区别只在于「打分的人」是真人还是另一个 AI。

对齐（Alignment）  
对齐是一个更宏观的概念：让模型的行为、价值观符合人类的期望和意图。  
它是 AI 安全领域的核心议题，贯穿在 SFT、RLHF 等各个训练环节中——SFT 和 RLHF 本质上都是「对齐手段」。  
    
微调（Fine-tuning）  
在已经训练好的通用模型基础上，用某个特定领域（如法律、医疗、客服）的数据继续训练。  
相比从零预训练，微调成本要低得多，是让通用模型适配专业场景的常用做法。  
    
MoE（混合专家模型）  
MoE 全称 Mixture of Experts，是一种模型架构设计。  
模型内部包含多个「专家」子网络，每次处理输入时只激活其中一部分，而不是让全部参数都参与计算。  
这样可以在扩大模型总规模的同时，控制实际计算量，做到「参数大、推理快」。   
   
![4](4_Training_Process.png)

## 开发者如何使用大模型
这是初学开发者最需要关注的部分：如何通过 API 让大模型为应用服务。  
本节先给一个最基础、可直接运行的对话调用示例，再展开各概念。  
    
最基础的 API 对话调用  
下面这段代码演示一次完整的对话请求，同时用到了 System Prompt、用户消息和温度三个概念。  

In [3]:
# 安装：pip install openai
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

# 创建客户端。OpenAI 兼容协议的服务都能复用这段代码，
# 换成其它服务商只需改 api_key 与 base_url
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),  # 从环境变量读取密钥（必填）
)

# 发起一次对话请求
response = client.chat.completions.create(
    model="deepseek-v4-flash",  # 模型名称（必填），按所用服务商更换
    messages=[
        # System Prompt：预先设定模型的角色与行为规范
        {"role": "system", "content": "你是一名专业的 Python 编程助手，只回答技术相关问题。"},
        # 用户消息：实际的问题
        {"role": "user", "content": "什么是 runoob？"},
    ],
    temperature=0.7,  # 生成温度，取值 0~2，越大越发散（见下文）
)

# 打印模型回答
print("回答：", response.choices[0].message.content)
# usage 字段记录了这次调用消耗的 Token 数（计费依据）
print("输入 Token 数：", response.usage.prompt_tokens)
print("输出 Token 数：", response.usage.completion_tokens)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

Prompt（提示词）  
Prompt 是你输入给模型的指令或问题，直接决定模型输出的质量。  
  
研究如何写出更清晰、更有效 Prompt 的技巧，叫做提示词工程（Prompt Engineering）。  
  
常见手法包括：给出具体示例、要求分步骤思考、指定输出格式等。  
  
System Prompt（系统提示词）  
在用户正式输入之前，预先设定给模型的指令，用来定义模型的角色、行为规范、背景知识。  
  
上文示例里的「你是一名专业的 Python 编程助手，只回答技术相关问题」就是典型的 System Prompt。  

System Prompt 与用户消息的区别：System Prompt 设定「模型该怎么做」，用户消息提出「具体要做什么」。

把通用规范放进 System Prompt，能让多轮对话的回答风格保持一致。

温度（Temperature）  
温度是控制模型输出随机性的参数，取值范围一般是 0 到 2。  
  
温度越高，回答越「有创意」、越随机，也越容易出错；温度越低，回答越保守、越稳定。

![Temperature](5_Temperature.png)

上下文学习（In-Context Learning）  
不需要对模型做任何微调，只要在 Prompt 里提供几个示例，模型就能「照猫画虎」完成新的类似任务。  
  
这是 Prompt Engineering 中最常用的技巧之一，也叫 Few-shot Learning（少样本学习）。  
  
思维链（Chain-of-Thought, CoT）  
引导模型把复杂问题拆解成一步步的推理过程，再给出最终答案，而不是直接跳到结论。  
  
实践证明，这种方式能显著提升模型在数学、逻辑推理等复杂任务上的准确率。  
  
最简单的做法是在 Prompt 中加一句「请一步步思考」。  
  
幻觉（Hallucination）  
幻觉指模型生成看似合理、语言流畅，但实际上错误或凭空编造的内容。  
  
这是 LLM 目前普遍存在的缺陷，原因是模型本质上在「续写最可能的文字」，而不是「查证事实」。  

缓解幻觉的核心手段之一就是下文的 RAG：先检索真实资料，再让模型基于资料作答。

让模型连接外部世界  
单纯的 LLM 只能依赖训练时学到的知识回答问题，既不知道最新信息，也无法主动做事。  
  
本节这些技术，就是解决这两个短板的关键。  
  
Embedding（嵌入）  
Embedding 把文字、图片等内容转换成一串数字（向量）。  
  
转换后，计算机就能通过计算向量之间的距离，判断两段内容在「语义」上是否相似。
  
这是搜索、推荐、RAG 等应用的底层基础。 
  
向量数据库  
向量数据库专门用来存储和检索 Embedding 向量，比如 Pinecone、Milvus、Chroma。  
  
它能在海量向量中快速找到与查询语义最相似的内容，是构建 RAG 系统必不可少的组件。  
  
RAG（检索增强生成）  
RAG 全称 Retrieval-Augmented Generation。  
  
它的工作流程是：先把外部知识库转成向量存进向量数据库；用户提问时检索出最相关的内容；再把检索结果连同问题交给模型生成答案。  
  
RAG 能有效缓解幻觉，并让模型回答训练数据之外的新知识。  

![RAG Retrieval-Augmented Generation Workflow](6%20RAG%20Retrieval-Augmented%20Generation%20Workflow.png)

下面是一个不依赖外部向量库、用 OpenAI Embedding 加 numpy 实现的最小 RAG 示例，方便理解原理：

In [ ]:
# 安装：pip install openai numpy
from openai import OpenAI
import numpy as np
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 模拟一个很小的"知识库"：几段文档
documents = [
    "runoob 是一个提供编程教程的中文学习网站。",
    "RAG 通过检索真实资料再生成，能有效减少大模型的幻觉。",
    "Python 使用缩进来表示代码块，通常建议每级缩进 4 个空格。",
]

# 第一步：把知识库向量化（离线建库，只需做一次）
doc_resp = client.embeddings.create(input=documents, model="text-embedding-3-small")
doc_vectors = [d.embedding for d in doc_resp.data]

# 第二步：把用户问题向量化
question = "什么是 runoob？"
query_vector = client.embeddings.create(
    input=question, model="text-embedding-3-small"
).data[0].embedding

# 第三步：用余弦相似度找出最相关的文档
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

scores = [cosine(query_vector, v) for v in doc_vectors]
best_doc = documents[int(np.argmax(scores))]  # 相似度最高的文档
print("检索到的资料：", best_doc)

# 第四步：把检索到的资料连同问题交给模型生成回答
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "请只根据下面提供的资料回答问题，资料中没有的信息不要编造。"},
        {"role": "user", "content": f"资料：{best_doc}\n\n问题：{question}"},
    ],
)
print("回答：", response.choices[0].message.content)

Function Calling（函数调用）  
Function Calling 让大模型能够调用外部工具或 API 来完成任务，比如查询实时天气、执行数学计算、查数据库、发送邮件。  
  
它弥补了模型「只能生成文字、不能执行动作」的缺陷，是大模型从聊天机器人进化为实用工具的关键一步。  
  
它的运作方式是两轮交互：第一轮模型决定要调用哪个函数、传什么参数；你的代码真正执行函数后，第二轮把结果回传，让模型生成自然语言回答。  

In [ ]:
from openai import OpenAI
import json
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 定义模型可调用的工具（函数）
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的实时天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称，例如：杭州"}
                },
                "required": ["city"],
            },
        },
    }
]

# 真正执行查询的本地函数（实际项目中替换为真实天气 API）
def get_weather(city):
    return f"{city} 今天晴，气温 28°C"

# 第一轮：把问题交给模型，模型决定调用哪个工具、传什么参数
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "杭州今天天气怎么样？"}],
    tools=tools,
)
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)  # 解析出参数，例如 {"city": "杭州"}
result = get_weather(args["city"])  # 真正执行工具
print("工具执行结果：", result)

# 第二轮：把工具结果回传给模型，让它基于结果生成自然语言回答
follow_up = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "杭州今天天气怎么样？"},
        response.choices[0].message,  # 模型上一步"调用工具"的决定
        {"role": "tool", "tool_call_id": tool_call.id, "content": result},  # 工具返回结果
    ],
    tools=tools,
)
print("最终回答：", follow_up.choices[0].message.content)

MCP（模型上下文协议）  
MCP 全称 Model Context Protocol，是一种让大模型标准化连接外部工具和数据源的开放协议。  
  
可以把它类比为「AI 领域的 USB 接口」：不同的工具和数据源只要遵循这个协议，就能被各种支持 MCP 的大模型统一调用。  
  
它解决的是 Function Calling 时代「每接一个工具都要重复对接」的问题，减少了开发工作量。  
  
Agent（智能体）  
Agent 是基于大模型构建的、能自主规划任务、调用工具、执行多步骤操作的系统。  
  
相比一问一答的对话，Agent 更像一个能独立完成复杂任务的「数字员工」，比如自动跑通「查资料 → 写代码 → 测试 → 修复 bug」整套流程。  
  
它依赖的核心循环是 ReAct（Reason + Act）：思考下一步、调用工具、观察结果、再思考，直到任务完成。  

![7](7%20Agent%20Work%20loop.png)

多模态（Multimodal）  
多模态指模型能同时理解和/或生成多种类型的数据，不局限于文本，还包括图像、音频、视频。  
  
现在很多主流大模型都已经具备读图、分析图表等多模态能力。  

![8](8%20Technical%20table.png)

让模型运行得更快、更省  
模型能力越强，往往意味着体积越大、运行成本越高。  
  
下面两个概念，就是为了让模型在实际部署时更轻量、更高效。  
  
量化（Quantization）  
量化把模型参数从高精度（比如 32 位浮点数）压缩到低精度（比如 8 位或 4 位整数）。  
  
它能大幅减少模型占用的显存、提升推理速度，代价是精度有一定损失，但很多场景下可以接受。  
  
模型蒸馏（Model Distillation）  
蒸馏用一个能力强但体积大的「教师模型」的输出，去训练一个体积更小的「学生模型」。  
  
目的是让小模型尽量学到大模型的能力，蒸馏后的模型体积更小、推理更快、部署成本更低。  

![9](9%20Comparison%20Items.png)

初学者学习路径  
对初学开发者来说，不需要一次性吃透所有概念。  
  
建议按照下面的顺序循序渐进，会比零散记概念更容易建立完整体系。

![10](10%20Learning%20Path%20for%20Beginners.png)

## 核心术语速查表
把全文出现的关键术语汇总成一张表，按「所属环节」分组，方便随时查阅回顾。

![-1](11%20Terminology-1.png)  
![-2](11%20Terminology-2.png)